# Retrieval Strategies — AI Engineering Interview Prep

BM25 (sparse), dense (semantic), and hybrid retrieval.

**Install**: `pip install rank-bm25 sentence-transformers faiss-cpu`

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import re

model = SentenceTransformer('all-MiniLM-L6-v2')
DIM = model.get_sentence_embedding_dimension()

# Corpus
corpus = [
    "Gradient descent minimizes loss by following the negative gradient direction iteratively.",
    "Adam optimizer adapts learning rates using first and second moment estimates of gradients.",
    "Stochastic gradient descent SGD updates weights using a random mini-batch each iteration.",
    "Overfitting happens when a model memorizes training data and fails on unseen test data.",
    "Dropout regularization randomly sets neuron activations to zero during training.",
    "L2 regularization adds a penalty proportional to the squared magnitude of weights.",
    "Batch normalization normalizes layer inputs to have zero mean and unit variance.",
    "Attention mechanisms compute weighted sums of values using query-key similarity scores.",
    "Self-attention allows each position in a sequence to attend to all other positions.",
    "BERT pre-trains using masked language modeling on bidirectional context.",
    "GPT uses autoregressive language modeling, predicting next tokens left to right.",
    "Transfer learning fine-tunes pretrained models on task-specific labeled data.",
    "Cross-entropy loss measures the divergence between predicted and true probability distributions.",
    "Precision measures the fraction of positive predictions that are truly positive.",
    "Recall measures the fraction of actual positives that the model correctly identifies.",
]

def tokenize(text):
    return re.findall(r'\w+', text.lower())

tokenized_corpus = [tokenize(doc) for doc in corpus]
corpus_embeddings = model.encode(corpus, normalize_embeddings=True).astype(np.float32)

print(f"Corpus: {len(corpus)} docs, dim={DIM}")

## 1. BM25 — Sparse Retrieval

In [ ]:
# BM25: TF-IDF variant with document length normalization
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, k=3):
    tokens = tokenize(query)
    scores = bm25.get_scores(tokens)
    top_k = np.argsort(scores)[::-1][:k]
    return [(corpus[i], scores[i]) for i in top_k]

queries = [
    "how does SGD gradient work",
    "preventing overfitting neural networks",
    "attention self-attention transformer",
]

for q in queries:
    print(f"\nBM25 query: '{q}'")
    for doc, score in bm25_search(q):
        print(f"  [{score:.4f}] {doc[:70]}...")

## 2. Dense Retrieval

In [ ]:
# Dense: FAISS inner product with normalized embeddings
index = faiss.IndexFlatIP(DIM)
index.add(corpus_embeddings)

def dense_search(query, k=3):
    q_emb = model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_emb, k)
    return [(corpus[i], float(scores[0][j])) for j, i in enumerate(indices[0])]

for q in queries:
    print(f"\nDense query: '{q}'")
    for doc, score in dense_search(q):
        print(f"  [{score:.4f}] {doc[:70]}...")

## 3. Hybrid Retrieval with Reciprocal Rank Fusion (RRF)

In [ ]:
def rrf_score(ranks, k=60):
    """Reciprocal Rank Fusion: sum 1/(k + rank) across retrievers."""
    return sum(1 / (k + r) for r in ranks)

def hybrid_search(query, k=3, top_n=10):
    # Get more candidates from each
    bm25_results = bm25_search(query, k=top_n)
    dense_results = dense_search(query, k=top_n)

    # Build rank dictionaries
    bm25_ranks = {doc: rank + 1 for rank, (doc, _) in enumerate(bm25_results)}
    dense_ranks = {doc: rank + 1 for rank, (doc, _) in enumerate(dense_results)}

    all_docs = set(bm25_ranks) | set(dense_ranks)
    rrf_scores = {}
    for doc in all_docs:
        ranks = []
        if doc in bm25_ranks: ranks.append(bm25_ranks[doc])
        else: ranks.append(top_n + 1)  # penalty for missing
        if doc in dense_ranks: ranks.append(dense_ranks[doc])
        else: ranks.append(top_n + 1)
        rrf_scores[doc] = rrf_score(ranks)

    top = sorted(rrf_scores.items(), key=lambda x: -x[1])[:k]
    return top

for q in queries:
    print(f"\nHybrid (RRF) query: '{q}'")
    for doc, score in hybrid_search(q):
        print(f"  [{score:.5f}] {doc[:70]}...")

## 4. Retrieval Evaluation

In [ ]:
# Evaluate with Recall@k and MRR@k
def recall_at_k(retrieved, relevant, k):
    retrieved_k = [d for d, _ in retrieved[:k]]
    hits = sum(1 for d in retrieved_k if d in relevant)
    return hits / len(relevant) if relevant else 0

def mrr_at_k(retrieved, relevant, k):
    for rank, (doc, _) in enumerate(retrieved[:k], 1):
        if doc in relevant:
            return 1 / rank
    return 0

# Define ground truth for test queries
test_cases = [
    {
        "query": "gradient descent optimization",
        "relevant": [corpus[0], corpus[1], corpus[2]]  # gradient, adam, sgd
    },
    {
        "query": "how to stop overfitting",
        "relevant": [corpus[3], corpus[4], corpus[5]]  # overfitting, dropout, L2
    },
    {
        "query": "attention mechanism in transformers",
        "relevant": [corpus[7], corpus[8], corpus[9]]  # attention, self-attention, BERT
    },
]

print(f"{'Retriever':<15} {'Recall@3':<12} {'MRR@3':<12}")
print("-" * 40)

for retriever_name, retriever_fn in [("BM25", bm25_search), ("Dense", dense_search), ("Hybrid", hybrid_search)]:
    recalls, mrrs = [], []
    for tc in test_cases:
        results = retriever_fn(tc["query"], k=3)
        recalls.append(recall_at_k(results, tc["relevant"], k=3))
        mrrs.append(mrr_at_k(results, tc["relevant"], k=3))
    print(f"{retriever_name:<15} {np.mean(recalls):.4f}       {np.mean(mrrs):.4f}")

## Interview Tips

- **BM25 strengths**: exact keyword matching, no GPU needed, great for technical terms/IDs.
- **Dense strengths**: handles synonyms, paraphrase, multilingual queries.
- **BM25 weaknesses**: fails on semantic similarity ('car' won't match 'automobile').
- **RRF**: simple hybrid fusion — no score normalization needed. `k=60` is the standard.
- **Recall@k**: fraction of relevant docs retrieved in top-k. Primary RAG metric.
- **MRR**: penalizes if relevant doc is ranked 3rd vs 1st. Good for single-answer lookups.
- **Production**: use dense for semantic, BM25 for filters/keywords, then re-rank top-50 with a cross-encoder.